# Feature Engineering for Citation Impact Prediction

This notebook creates features for modeling based on EDA insights:
- **Text Features**: TF-IDF from abstracts (200-300 features)
- **Collaboration Features**: Author count categories
- **Temporal Features**: Years since publication
- **Venue Features**: Handle missing values
- **Target Variables**: Citations_log (regression), HighImpact (classification)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import pickle
import json
from datetime import datetime

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries loaded successfully!")

## 1. Load Merged Dataset

In [ ]:
# Load the merged dataset from data_merge.ipynb
df = pd.read_csv('../data/merged_citation_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check for target variables
print("Target Variables:")
print(f"  Citations_log: {'✓' if 'Citations_log' in df.columns else '✗ MISSING'}")
print(f"  HighImpact: {'✓' if 'HighImpact' in df.columns else '✗ MISSING'}")

if 'HighImpact' in df.columns:
    print(f"\nClass distribution:")
    print(df['HighImpact'].value_counts())
    print(f"\nHighImpact threshold: {df[df['HighImpact']==1]['Citations'].min()} citations")

## 2. Text Features: TF-IDF from Abstracts

**Strategy based on EDA:**
- Abstract length has NO correlation with citations (-0.033)
- Abstract *content* likely matters (terminology, topics)
- Extract 200-300 TF-IDF features
- Use unigrams + bigrams for better context

In [ ]:
# Check for missing abstracts
df['Abstract'] = df['Abstract'].fillna('')  # Fill NaN with empty string
df['abstract_length'] = df['Abstract'].str.len()

print("Abstract Statistics:")
print(f"  Total papers: {len(df)}")
print(f"  Papers with abstracts: {(df['abstract_length'] > 0).sum()}")
print(f"  Papers without abstracts: {(df['abstract_length'] == 0).sum()}")
print(f"\n  Mean abstract length: {df[df['abstract_length'] > 0]['abstract_length'].mean():.0f} chars")
print(f"  Median abstract length: {df[df['abstract_length'] > 0]['abstract_length'].median():.0f} chars")

In [ ]:
# Create TF-IDF features
print("Extracting TF-IDF features from abstracts...")

# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=300,           # Top 300 most important terms
    ngram_range=(1, 2),         # Unigrams and bigrams
    min_df=5,                   # Term must appear in at least 5 documents
    max_df=0.8,                 # Term must appear in less than 80% of documents
    stop_words='english',       # Remove common English words
    lowercase=True,
    strip_accents='unicode'
)

# Fit and transform abstracts
tfidf_matrix = tfidf.fit_transform(df['Abstract'])

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{term}' for term in tfidf.get_feature_names_out()]
)

print(f"\n✓ Created {tfidf_df.shape[1]} TF-IDF features")
print(f"\nTop 20 most important terms:")
print(tfidf_df.columns[:20].tolist())

In [ ]:
# Save TF-IDF vectorizer for future use (deployment)
import os
os.makedirs('../models', exist_ok=True)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("✓ TF-IDF vectorizer saved to models/tfidf_vectorizer.pkl")

## 3. Collaboration Features

**Strategy based on EDA:**
- Mega-collaborations (50+ authors): 55 median citations (18x boost!)
- Large teams (11-50): 16 median citations
- Medium/Small teams (2-10): 9-10 median citations
- Single-author: 3 median citations
- **Non-linear relationship → use categorical features**

In [ ]:
# Create collaboration size categories
def categorize_collaboration(n_authors):
    if n_authors == 1:
        return 'single'
    elif n_authors <= 5:
        return 'small'
    elif n_authors <= 10:
        return 'medium'
    elif n_authors <= 50:
        return 'large'
    else:
        return 'mega'

df['collab_size'] = df['Number of Authors'].apply(categorize_collaboration)

print("Collaboration Size Distribution:")
print(df['collab_size'].value_counts().sort_index())

# Create binary flag for mega-collaborations (strong signal)
df['is_mega_collab'] = (df['Number of Authors'] >= 50).astype(int)

print(f"\nMega-collaborations: {df['is_mega_collab'].sum()} ({df['is_mega_collab'].mean()*100:.1f}%)")

In [ ]:
# One-hot encode collaboration categories
collab_dummies = pd.get_dummies(df['collab_size'], prefix='collab', drop_first=True)

print(f"✓ Created {len(collab_dummies.columns)} collaboration category features")
print(f"  Features: {collab_dummies.columns.tolist()}")

## 4. Temporal Features

**Strategy based on EDA:**
- Year correlation with Citations_log: **-0.38** (strongest predictor!)
- Older papers have more citations due to time
- Create "years since publication" to control for this bias

In [ ]:
# Get current year for age calculation
current_year = datetime.now().year

# Create temporal features
df['years_since_publication'] = current_year - df['Year']
df['publication_age_log'] = np.log1p(df['years_since_publication'])  # Log-transform for skew

print(f"Temporal Features (current year: {current_year}):")
print(f"\nYears since publication:")
print(df['years_since_publication'].describe())

print(f"\nYear range: {df['Year'].min()} - {df['Year'].max()}")
print(f"Age range: {df['years_since_publication'].min()} - {df['years_since_publication'].max()} years")

In [ ]:
# Check correlation with citations
if 'Citations_log' in df.columns:
    corr_year = df[['Year', 'years_since_publication', 'publication_age_log', 'Citations_log']].corr()['Citations_log']
    print("\nCorrelation with Citations_log:")
    print(corr_year)

## 5. Venue Features: Handle Missing Values

**Strategy based on EDA:**
- SNIP: 9.2% missing
- CiteScore: 12.8% missing
- SJR: 9.8% missing
- High multicollinearity (0.78-0.95)
- SJR has strongest correlation (0.38 with Citations_log)

**Approach**: Median imputation (conservative)

In [ ]:
# Check missing values in venue metrics
venue_cols = ['SNIP (publication year)', 'CiteScore (publication year)', 'SJR (publication year)']
venue_cols_exist = [col for col in venue_cols if col in df.columns]

print("Missing Values in Venue Metrics:")
for col in venue_cols_exist:
    missing = df[col].isna().sum()
    print(f"  {col}: {missing} ({missing/len(df)*100:.1f}%)")

In [ ]:
# Convert venue metrics to numeric and impute missing values with median
print("\nConverting venue metrics to numeric and imputing with median...")

for col in venue_cols_exist:
    # Convert to numeric, coercing errors (e.g., '-', 'N/A') to NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Count missing after conversion
    missing_after = df[col].isna().sum()
    
    # Impute with median
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    
    print(f"  {col}: {missing_after} missing → filled with {median_value:.2f}")

# Verify no missing values remain
print("\n✓ Verification:")
for col in venue_cols_exist:
    print(f"  {col}: {df[col].isna().sum()} missing")

In [ ]:
# Create simplified column names for modeling
df['venue_snip'] = df.get('SNIP (publication year)', 0)
df['venue_citescore'] = df.get('CiteScore (publication year)', 0)
df['venue_sjr'] = df.get('SJR (publication year)', 0)

print("✓ Created clean venue feature columns: venue_snip, venue_citescore, venue_sjr")

## 6. Combine All Features

In [ ]:
# Select base features (non-text)
base_features = [
    'Number of Authors',           # Raw author count
    'is_mega_collab',              # Binary mega-collaboration flag
    'years_since_publication',     # Temporal feature
    'publication_age_log',         # Log-transformed age
    'venue_snip',                  # Venue prestige metrics
    'venue_citescore',
    'venue_sjr',
]

# Create feature matrix
X_base = df[base_features].copy()
X_collab = collab_dummies.copy()
X_text = tfidf_df.copy()

# Combine all features
X = pd.concat([X_base, X_collab, X_text], axis=1)

print(f"Feature Matrix Shape: {X.shape}")
print(f"\nFeature Breakdown:")
print(f"  Base features: {len(base_features)}")
print(f"  Collaboration categories: {len(collab_dummies.columns)}")
print(f"  TF-IDF features: {len(tfidf_df.columns)}")
print(f"  TOTAL FEATURES: {X.shape[1]}")

In [ ]:
# Check for missing values in feature matrix
missing_counts = X.isna().sum()
if missing_counts.sum() > 0:
    print("⚠️  WARNING: Missing values detected:")
    print(missing_counts[missing_counts > 0])
else:
    print("✓ No missing values in feature matrix")

## 7. Target Variables

In [ ]:
# Extract target variables
y_regression = df['Citations_log'].copy()  # For regression
y_classification = df['HighImpact'].copy()  # For classification

print("Target Variables:")
print(f"\nRegression (Citations_log):")
print(y_regression.describe())

print(f"\nClassification (HighImpact):")
print(y_classification.value_counts())
print(f"  Class balance: {y_classification.value_counts(normalize=True).to_dict()}")

## 8. Train-Test Split with Temporal Stratification

**Strategy:**
- Use temporal split to test generalization to newer papers
- Train on older papers (2010-2020)
- Test on recent papers (2021+)
- This mimics real-world deployment scenario

In [ ]:
# Check year distribution
print("Year Distribution:")
print(df['Year'].value_counts().sort_index())
print(f"\nMedian year: {df['Year'].median():.0f}")

In [ ]:
# Temporal split: train on 2010-2020, test on 2021+
temporal_cutoff = 2021

train_mask = df['Year'] < temporal_cutoff
test_mask = df['Year'] >= temporal_cutoff

X_train_temporal = X[train_mask]
X_test_temporal = X[test_mask]
y_reg_train_temporal = y_regression[train_mask]
y_reg_test_temporal = y_regression[test_mask]
y_cls_train_temporal = y_classification[train_mask]
y_cls_test_temporal = y_classification[test_mask]

print(f"Temporal Split (cutoff: {temporal_cutoff}):")
print(f"  Train: {X_train_temporal.shape[0]} papers (before {temporal_cutoff})")
print(f"  Test: {X_test_temporal.shape[0]} papers ({temporal_cutoff}+)")
print(f"\n  Train class balance: {y_cls_train_temporal.value_counts(normalize=True).to_dict()}")
print(f"  Test class balance: {y_cls_test_temporal.value_counts(normalize=True).to_dict()}")

In [ ]:
# Also create random stratified split for comparison
X_train_random, X_test_random, y_reg_train_random, y_reg_test_random, y_cls_train_random, y_cls_test_random = train_test_split(
    X, y_regression, y_classification,
    test_size=0.2,
    stratify=y_classification,  # Maintain class balance
    random_state=42
)

print(f"\nRandom Stratified Split (80/20):")
print(f"  Train: {X_train_random.shape[0]} papers")
print(f"  Test: {X_test_random.shape[0]} papers")
print(f"\n  Train class balance: {y_cls_train_random.value_counts(normalize=True).to_dict()}")
print(f"  Test class balance: {y_cls_test_random.value_counts(normalize=True).to_dict()}")

## 9. Save Processed Data

In [ ]:
# Save feature matrix and targets
print("Saving processed datasets...")

# Full dataset with features
X.to_csv('../data/features_matrix.csv', index=False)
y_regression.to_csv('../data/target_regression.csv', index=False, header=['Citations_log'])
y_classification.to_csv('../data/target_classification.csv', index=False, header=['HighImpact'])

print("✓ Saved:")
print("  - data/features_matrix.csv")
print("  - data/target_regression.csv")
print("  - data/target_classification.csv")

In [ ]:
# Save temporal splits
X_train_temporal.to_csv('../data/X_train_temporal.csv', index=False)
X_test_temporal.to_csv('../data/X_test_temporal.csv', index=False)
y_reg_train_temporal.to_csv('../data/y_reg_train_temporal.csv', index=False, header=['Citations_log'])
y_reg_test_temporal.to_csv('../data/y_reg_test_temporal.csv', index=False, header=['Citations_log'])
y_cls_train_temporal.to_csv('../data/y_cls_train_temporal.csv', index=False, header=['HighImpact'])
y_cls_test_temporal.to_csv('../data/y_cls_test_temporal.csv', index=False, header=['HighImpact'])

print("\n✓ Saved temporal splits (train: <2021, test: 2021+)")

In [ ]:
# Save random splits
X_train_random.to_csv('../data/X_train_random.csv', index=False)
X_test_random.to_csv('../data/X_test_random.csv', index=False)
y_reg_train_random.to_csv('../data/y_reg_train_random.csv', index=False, header=['Citations_log'])
y_reg_test_random.to_csv('../data/y_reg_test_random.csv', index=False, header=['Citations_log'])
y_cls_train_random.to_csv('../data/y_cls_train_random.csv', index=False, header=['HighImpact'])
y_cls_test_random.to_csv('../data/y_cls_test_random.csv', index=False, header=['HighImpact'])

print("✓ Saved random stratified splits (80/20)")

In [ ]:
# Save feature metadata
feature_info = {
    'total_features': X.shape[1],
    'feature_names': X.columns.tolist(),
    'base_features': base_features,
    'collaboration_features': collab_dummies.columns.tolist(),
    'tfidf_features': tfidf_df.columns.tolist(),
    'n_base': len(base_features),
    'n_collaboration': len(collab_dummies.columns),
    'n_tfidf': len(tfidf_df.columns),
    'temporal_cutoff': temporal_cutoff,
    'train_size_temporal': X_train_temporal.shape[0],
    'test_size_temporal': X_test_temporal.shape[0],
    'train_size_random': X_train_random.shape[0],
    'test_size_random': X_test_random.shape[0]
}

with open('../models/feature_info.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

print("\n✓ Saved feature metadata to models/feature_info.json")

## 10. Summary

In [ ]:
print("="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)

print(f"\n📊 DATASET:")
print(f"  Total papers: {X.shape[0]:,}")
print(f"  Total features: {X.shape[1]:,}")

print(f"\n🔧 FEATURES CREATED:")
print(f"  ✓ Base features: {len(base_features)}")
print(f"     - Author count")
print(f"     - Mega-collaboration flag")
print(f"     - Years since publication")
print(f"     - Venue metrics (SNIP, CiteScore, SJR)")
print(f"  ✓ Collaboration categories: {len(collab_dummies.columns)}")
print(f"  ✓ TF-IDF text features: {len(tfidf_df.columns)}")

print(f"\n🎯 TARGETS:")
print(f"  ✓ Regression: Citations_log (mean={y_regression.mean():.2f}, std={y_regression.std():.2f})")
print(f"  ✓ Classification: HighImpact (class balance: {y_classification.value_counts(normalize=True)[1]*100:.1f}% / {y_classification.value_counts(normalize=True)[0]*100:.1f}%)")

print(f"\n📂 FILES SAVED:")
print(f"  ✓ data/features_matrix.csv")
print(f"  ✓ data/target_regression.csv")
print(f"  ✓ data/target_classification.csv")
print(f"  ✓ Temporal splits (6 files)")
print(f"  ✓ Random splits (6 files)")
print(f"  ✓ models/tfidf_vectorizer.pkl")
print(f"  ✓ models/feature_info.json")

print(f"\n🚀 NEXT STEPS:")
print(f"  1. Train baseline models (Logistic Regression, Linear Regression)")
print(f"  2. Train tree-based models (Random Forest, XGBoost, LightGBM)")
print(f"  3. Compare temporal vs random split performance")
print(f"  4. Feature importance analysis")
print(f"  5. Hyperparameter tuning on best models")

print("\n" + "="*80)